# Latent factor-based Collaborative Filtering Recommender Systems
**Prepared by Christian Alis**

In [ ]:
import numpy as np
import pandas as pd
from numpy.testing import (
    assert_equal,
    assert_array_equal,
    assert_array_almost_equal,
    assert_allclose,
)

You may have noticed that when we perform collaborative filtering, what we are doing is actually completing the matrix. Being a matrix, we can then decompose the utility matrix $\mathbf{M}$ into two matrices: $\mathbf{M} = \mathbf{F}_{user} \mathbf{F}_{item}^T$. $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ are basically the UV-decomposition of $\mathbf{M}$. If $\mathbf{M}$ is a $n$-user $\times$ $k$-item matrix then $\mathbf{F}_{user}$ is $n \times d$ and $\mathbf{F}_{item}$ is $k \times d$. The dimension $d$ is the number of latent components or latent factors.

If we have the $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ matrices then by multiplying them, we are able to recreate $\mathbf{M}$. Put another way $\mathbf{F}_{user}\mathbf{F}_{item}^T$ completes the matrix $\mathbf{M}$. Our goal therefore is to find $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ to recreate the observed ratings. We can express this as the objective function,
$$
J = \frac{1}{2} \left|\left| \mathbf{M} - \mathbf{F}_{user}\mathbf{F}_{item}^T \right|\right|^2.
$$
Here, we are using the Frobenius norm as the distance measure. We also only consider the entries that have been rated. Let $u_{ij}$, $v_{ij}$ and $r_{ij}$ be the elements of $\mathbf{F}_{user}$ ($\mathbf U$), $\mathbf{F}_{item}$ ($\mathbf V$) and $\mathbf M$, respectively. We can then rewrite the objective function as
$$
J = \frac{1}{2} \sum_{(i,j): r_{ij} \neq \emptyset} \left( r_{ij} - \sum_{s=1}^d u_{is} v_{js} \right)^2.
$$
As expected, we can use gradient descent to minimize the objective function. The partial derivatives or gradient is given by
$$
\begin{align}
\frac{\partial J}{\partial u_{iq}} & = \sum_{j: r_{ij} \neq \emptyset} \left( r_{ij} - \sum_{s=1}^d u_{is} v_{js} \right) \left( -v_{jq} \right) \forall i \in \left\{ 1...n \right\}, q \in \left\{ 1...d \right\},\\
\frac{\partial J}{\partial v_{jq}} & = \sum_{i: r_{ij} \neq \emptyset} \left( r_{ij} - \sum_{s=1}^d u_{is} v_{js} \right) \left( -u_{iq} \right) \forall j \in \left\{ 1...k \right\}, q \in \left\{ 1...d \right\}.
\end{align}
$$
If we let $e_{ij} = r_{ij} - \sum_{s=1}^d u_{is} v_{js}$, the partial derivatives can be further simplified to
$$
\begin{align}
\frac{\partial J}{\partial u_{iq}} & = \sum_{j: r_{ij} \neq \emptyset} \left( e_{ij} \right) \left( -v_{jq} \right) \forall i \in \left\{ 1...n \right\}, q \in \left\{ 1...d \right\},\\
\frac{\partial J}{\partial u_{jq}} & = \sum_{i: r_{ij} \neq \emptyset} \left( e_{ij} \right) \left( -u_{iq} \right) \forall j \in \left\{ 1...k \right\}, q \in \left\{ 1...d \right\},
\end{align}
$$
and written as
$$
\begin{align}
\mathbf{U} & \leftarrow \mathbf{U} + \alpha \mathbf{EV}; \\
\mathbf{V} & \leftarrow \mathbf{V} + \alpha \mathbf{E}^T \mathbf{U}
\end{align}
$$
in matrix form where $\alpha > 0$ is the step size. We can write the algorithm as

**Algorithm** *GD*(Ratings Matrix: $R$, Learning Rate: $\alpha$)  
**begin**  
$\quad$Randomly initialize matrices $U$ and $V$;  
$\quad$$S=\{(i, j): r_{ij}\text{ is observed}\}$;  
$\quad$**while** not(convergence) **do**  
$\quad$**begin**\
$\quad$$\quad$Compute each error $e_{ij} \in S$ as the observed entries of $R-UV^T$;  
$\quad$$\quad$**for** each user-component pair $(i,q)$ **do** $u_{iq}^+ \Leftarrow u_{iq} + \alpha \cdot \sum_{j:(i,j) \in S} e_{ij} \cdot v_{jq}$;  
$\quad$$\quad$**for** each item-component pair $(j,q)$ **do** $v_{jq}^+ \Leftarrow v_{jq} + \alpha \cdot \sum_{i:(i,j) \in S} e_{ij} \cdot u_{iq}$;  
$\quad$$\quad$**for** each user-component pair $(i,q)$ **do** $u_{iq} \Leftarrow u_{iq}^+$;  
$\quad$$\quad$**for** each item-component pair $(j,q)$ **do** $v_{jq} \Leftarrow v_{jq}^+$;  
$\quad$$\quad$Check convergence condition;  
$\quad$**end**  
**end**

The algorithm above is the batch update method and requires to go through all of the known ratings at every iteration. An improvement is to randomly sample ratings and use that sample for measuring the error at every iteration--stochastic gradient descent. In this notebook, we look at two methods of efficiently performing UV decomposition on utility matrices that are being used in practice.

## Alternating least squares

Alternating least squares (ALS) is a more stable approach than SGD. From its name, it involves performing least squares regression alternately on $\mathbf U$ and $\mathbf V$. That is, we repeat the following two steps until convergence:

1. With $\mathbf U$ fixed, solve for each of the $d$ rows of $\mathbf V$ by using them as regressors to predict a known rating. Recall that $r_{ij} = \sum_{s=1}^d u_{is} v_{js}$. To determine the optimal vector $\mathbf{v}_j$ for the $j$th row of $\mathbf V$, we need to minimize $\sum_{i: r_{ij} \neq \emptyset} \left( r_{ij} - \sum_{s=1}^d u_{is} v_{js} \right)^2$ keeping $u_{i1}...u_{in}$ constant. We do it for each of the $d$ rows which gives us $d$ least-squares problems that are independent of each other making their solution parallelizable.

2. With $\mathbf V$ fixed, solve for each of the $n$ rows of $\mathbf U$ by using them as regressors to predict a known rating. Recall that $r_{ij} = \sum_{s=1}^k u_{is} v_{js}$. To determine the optimal vector $\mathbf{u}_i$ for the $i$th row of $\mathbf U$, we need to minimize $\sum_{j: r_{ij} \neq \emptyset} \left( r_{ij} - \sum_{s=1}^d u_{is} v_{js} \right)^2$ keeping $v_{j1}...v_{jd}$ constant. We do it for each of the $n$ rows which gives us $n$ least-squares problems that are independent of each other making their solution parallelizable.

To illustrate, suppose the utility matrix is given as
$$
\left(
\matrix{
5 & 2 & 4 & 4 & 3 \\
3 & 1 & 2 & 4 & 1 \\
  &   & 3 & 1 & 4 \\
2 & 5 & 4 & 3 & 5 \\
4 & 4 & 5 & 4 & 
}
\right).
$$

We want to decompose it into two matrices $\mathbf U$ and $\mathbf V$ using ALS. Suppose we begin with setting all elements of $\mathbf U$ to 1, we would then get:

$$
\left(
\matrix{
1 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1
}\right)
\times
\left(
\matrix{
v_{11} & v_{21} & v_{31} & v_{41} & v_{51} \\
v_{12} & v_{22} & v_{32} & v_{42} & v_{52}
}
\right)
=
\left(
\matrix{
v_{11} + v_{12} & v_{21} + v_{22} & v_{31} + v_{32} & v_{41} + v_{42} & v_{51} + v_{52} \\
v_{11} + v_{12} & v_{21} + v_{22} & v_{31} + v_{32} & v_{41} + v_{42} & v_{51} + v_{52} \\
v_{11} + v_{12} & v_{21} + v_{22} & v_{31} + v_{32} & v_{41} + v_{42} & v_{51} + v_{52} \\
v_{11} + v_{12} & v_{21} + v_{22} & v_{31} + v_{32} & v_{41} + v_{42} & v_{51} + v_{52} \\
v_{11} + v_{12} & v_{21} + v_{22} & v_{31} + v_{32} & v_{41} + v_{42} & v_{51} + v_{52}
}
\right)
\approx
\left(
\matrix{
5 & 2 & 4 & 4 & 3 \\
3 & 1 & 2 & 4 & 1 \\
  &   & 3 & 1 & 4 \\
2 & 5 & 4 & 3 & 5 \\
4 & 4 & 5 & 4 & 
}
\right)
$$
The resulting $d=5$ functions that we need to minimize are:
$$
\left[ 5 - \left( v_{11} + v_{12} \right) \right]^2 + \left[ 3 - \left( v_{11} + v_{12} \right) \right]^2 + \left[ 2 - \left( v_{11} + v_{12} \right) \right]^2 + \left[ 4 - \left( v_{11} + v_{12} \right) \right]^2; \\
\left[ 2 - \left( v_{21} + v_{22} \right) \right]^2 + \left[ 1 - \left( v_{21} + v_{22} \right) \right]^2 + \left[ 5 - \left( v_{21} + v_{22} \right) \right]^2 + \left[ 4 - \left( v_{21} + v_{22} \right) \right]^2; \\
\left[ 4 - \left( v_{31} + v_{32} \right) \right]^2 + \left[ 2 - \left( v_{31} + v_{32} \right) \right]^2 + \left[ 3 - \left( v_{31} + v_{32} \right) \right]^2 + \left[ 4 - \left( v_{31} + v_{32} \right) \right]^2 + \left[ 5 - \left( v_{31} + v_{32} \right) \right]^2; \\
\left[ 4 - \left( v_{41} + v_{42} \right) \right]^2 + \left[ 4 - \left( v_{41} + v_{42} \right) \right]^2 + \left[ 1 - \left( v_{41} + v_{42} \right) \right]^2 + \left[ 3 - \left( v_{41} + v_{42} \right) \right]^2 + \left[ 4 - \left( v_{41} + v_{42} \right) \right]^2; \\
\left[ 3 - \left( v_{51} + v_{52} \right) \right]^2 + \left[ 1 - \left( v_{51} + v_{52} \right) \right]^2 + \left[ 4 - \left( v_{51} + v_{52} \right) \right]^2 + \left[ 5 - \left( v_{51} + v_{52} \right) \right]^2 
$$
Doing the minimization we would get
$$
\mathbf{V}^T =
\left(
\matrix{
1.75 & 1.5 & 1.8 & 1.6 & 1.625 \\
1.75 & 1.5 & 1.8 & 1.6 & 1.625
}
\right).
$$
This will take us to the next step:
$$
\left(
\matrix{
u_{11} & u_{12} \\
u_{21} & u_{22} \\
u_{31} & u_{32} \\
u_{41} & u_{42} \\
u_{51} & u_{52}
}\right)
\times
\left(
\matrix{
1.75 & 1.5 & 1.8 & 1.6 & 1.625 \\
1.75 & 1.5 & 1.8 & 1.6 & 1.625
}
\right)
=
\left(
\matrix{
1.75u_{11} + 1.75u_{12} & 1.5u_{11} + 1.5u_{12} & 1.8u_{11} + 1.8u_{12} & 1.6u_{11} + 1.6u_{12} & 1.625u_{11} + 1.625u_{12} \\
1.75u_{21} + 1.75u_{22} & 1.5u_{21} + 1.5u_{22} & 1.8u_{21} + 1.8u_{22} & 1.6u_{21} + 1.6u_{22} & 1.625u_{21} + 1.625u_{22} \\
1.75u_{31} + 1.75u_{32} & 1.5u_{31} + 1.5u_{32} & 1.8u_{31} + 1.8u_{32} & 1.6u_{31} + 1.6u_{32} & 1.625u_{31} + 1.625u_{32} \\
1.75u_{41} + 1.75u_{42} & 1.5u_{41} + 1.5u_{42} & 1.8u_{41} + 1.8u_{42} & 1.6u_{41} + 1.6u_{42} & 1.625u_{41} + 1.625u_{42} \\
1.75u_{51} + 1.75u_{52} & 1.5u_{51} + 1.5u_{52} & 1.8u_{51} + 1.8u_{52} & 1.6u_{51} + 1.6u_{52} & 1.625u_{51} + 1.625u_{52}
}
\right)
\approx
\left(
\matrix{
5 & 2 & 4 & 4 & 3 \\
3 & 1 & 2 & 4 & 1 \\
  &   & 3 & 1 & 4 \\
2 & 5 & 4 & 3 & 5 \\
4 & 4 & 5 & 4 & 
}
\right)
$$
Giving us the following $n=5$ functions to minimize:
$$
\left[ 5 - \left( 1.75u_{11} + 1.75u_{12} \right) \right]^2 + \left[ 2 - \left( 1.5u_{11} + 1.5u_{12} \right) \right]^2 + \left[ 4 - \left( 1.8u_{11} + 1.8u_{12} \right) \right]^2 + \left[ 4 - \left( 1.6u_{11} + 1.6u_{12} \right) \right]^2 + \left[ 3 - \left( 1.625u_{11} + 1.625u_{12} \right) \right]^2; \\
\left[ 3 - \left( 1.75u_{21} + 1.75u_{22} \right) \right]^2 + \left[ 1 - \left( 1.5u_{21} + 1.5u_{22} \right) \right]^2 + \left[ 2 - \left( 1.8u_{21} + 1.8u_{22} \right) \right]^2 + \left[ 4 - \left( 1.6u_{21} + 1.6u_{22} \right) \right]^2 + \left[ 1 - \left( 1.625u_{21} + 1.625u_{22} \right) \right]^2; \\
\left[ 3 - \left( 1.8u_{31} + 1.8u_{32} \right) \right]^2 + \left[ 1 - \left( 1.6u_{31} + 1.6u_{32} \right) \right]^2 + \left[ 4 - \left( 1.625u_{31} + 1.625u_{32} \right) \right]^2; \\
\left[ 2 - \left( 1.75u_{41} + 1.75u_{42} \right) \right]^2 + \left[ 5 - \left( 1.5u_{41} + 1.5u_{42} \right) \right]^2 + \left[ 4 - \left( 1.8u_{41} + 1.8u_{42} \right) \right]^2 + \left[ 3 - \left( 1.6u_{41} + 1.6u_{42} \right) \right]^2 + \left[ 5 - \left( 1.625u_{41} + 1.625u_{42} \right) \right]^2; \\
\left[ 4 - \left( 1.75u_{51} + 1.75u_{52} \right) \right]^2 + \left[ 4 - \left( 1.5u_{51} + 1.5u_{52} \right) \right]^2 + \left[ 5 - \left( 1.8u_{51} + 1.8u_{52} \right) \right]^2 + \left[ 4 - \left( 1.6u_{51} + 1.6u_{52} \right) \right]^2;
$$
This would give us
$$
\mathbf{U} =
\left(
\matrix{
1.09884117 & 1.09884117 \\
0.66802999 & 0.66802999 \\
0.79970381 & 0.79970381 \\
1.13156101 & 1.13156101 \\
1.27784027 & 1.27784027
}\right).
$$
We then check whether the convergence condition (SSE, MSE, RMSE, etc.) has been satisfied. If not yet, then we repeat the two steps.

The function `als` below accepts the utility matrix `M`, the number of latent factors `d` to consider and tolerance `tol` then returns the $\mathbf{F}_\text{user}$ and $\mathbf{F}_\text{item}$ matrices when the RMSE is less than `tol`. For the sake of consistency for the asserts, both matrices were initially set to be all ones.

In [ ]:
def als(M, d, tol):
    from sklearn.linear_model import LinearRegression
    from scipy.linalg import lstsq
    M = np.asarray(M)
    U = np.ones((M.shape[0], d))
    V = np.ones((M.shape[1], d))
    while True:
        for j in range(V.shape[0]):
            has_rating = np.isfinite(M[:,j])
            V[j,:] = lstsq(
                U[has_rating,:],
                M[:,j][has_rating],
            )[0]
        for i in range(U.shape[0]):
            has_rating = np.isfinite(M[i,:])
            U[i,:] = lstsq(
                V[has_rating,:],
                M[i,:][has_rating],
            )[0]
        rmse = np.sqrt(np.nanmean((M - U@V.T)**2))
        if rmse < tol:
            return U, V

**Problem 1**

Create a function `recommend_als` that accepts the index of the user, utility matrix, $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ then returns a list of `L` recommended unrated items to `user` sorted from most recommended to least then by joke id.

In [ ]:
def recommend_als(user, df_utility, f_user, f_item, L):
    """Recommend unrated items to a user using ALS factor matrices.
    
    Parameters
    ----------
    user : int
        Positional index of the user (row) in ``df_utility``.
    df_utility : pandas.DataFrame
        Utility matrix of ratings, with missing ratings as NaN.
    f_user : numpy.ndarray
        User factor matrix, shape (n_users, d).
    f_item : numpy.ndarray
        Item factor matrix, shape (n_items, d).
    L : int
        Number of items to recommend.

    Returns
    -------
    list
        Up to L item (column) labels the user has not rated,
        sorted by predicted rating descending, ties broken by
        ascending item label.
    """
    preds = f_user[user, :] @ f_item.T
    preds = pd.Series(preds, index=df_utility.columns)

    rated = df_utility.iloc[user].notna()
    unrated_preds = preds[~rated]
    
    unrated_preds = unrated_preds.sort_index()
    unrated_preds = unrated_preds.sort_values(ascending=False, kind="stable")

    return unrated_preds.index[:L].tolist()

In [ ]:
df_jester = pd.read_excel(
    "/mnt/data/public/jester/dataset1/jester-data-2.xls",
    header=None,
    nrows=100,
).iloc[:, 1:]
df_jester.replace(99, np.nan, inplace=True)
f_user, f_item = als(df_jester, 50, 2)
recos = recommend_als(5, df_jester, f_user, f_item, 10)
assert_equal(recos, [51, 55, 1, 70, 30, 63, 9, 52, 43, 59])

## Coordinate gradient descent

Although ALS is parallelizable, it is not as efficient as stochastic gradient descent. Coordinate gradient descent (CGD) tries to address this tradeoff between stability and efficiently. Recall that in ALS, we fix one of the factor matrix then optimize the other. In CGD, we fix both matrices except for one element in one of them, then optimize that single element.

To decompose $\mathbf M$, what we can do is to start with two random $n \times d$ and $k \times d$ matrices corresponding to $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$, respectively. We pick an element in $\mathbf{F}_{user}$ or $\mathbf{F}_{item}$ then optimize that element such that the RMSE of the known values with the resulting values is minimized. We repeat this with another random element until the improvement in RMSE is below a certain threshold.

To illustrate, consider the matrices below.

$$
\left(
\matrix{
1 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1
}\right)
\times
\left(
\matrix{
1 & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 & 1
}
\right)
=
\left(
\matrix{
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2
}
\right)
\approx
\left(
\matrix{
5 & 2 & 4 & 4 & 3 \\
3 & 1 & 2 & 4 & 1 \\
  &   & 3 & 1 & 4 \\
2 & 5 & 4 & 3 & 5 \\
4 & 4 & 5 & 4 & 
}
\right)
$$

The matrices on the left are the $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ matrices, and the matrix on the middle is the product of $\mathbf{F}_{user}$ and $\mathbf{F}_{item}^T$. The matrix on the right is the utility matrix $\mathbf M$ which we would like to match. Notice that $\mathbf M$ has empty elements. We've also "randomly" initialized $\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ to all ones.

Suppose we want to estimate the value of the upper-leftmost element of $\mathbf{F}_{user}$, we would get:

$$
\left(
\matrix{
x & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1
}\right)
\times
\left(
\matrix{
1 & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 & 1
}
\right)
=
\left(
\matrix{
x+1 & x+1 & x+1 & x+1 & x+1 \\
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2 \\
2 & 2 & 2 & 2 & 2
}
\right)
\approx
\left(
\matrix{
5 & 2 & 4 & 4 & 3 \\
3 & 1 & 2 & 4 & 1 \\
  &   & 3 & 1 & 4 \\
2 & 5 & 4 & 3 & 5 \\
4 & 4 & 5 & 4 & 
}
\right)
$$

Notice that only the first row of $\mathbf M$ was affected. The contribution of this row to the SSE is

$$(5 - (x+1))^2 + (2  - (x+1))^2 + (4  - (x+1))^2 + (4  - (x+1))^2 + (3 - (x+1))^2 \\ 
= (4-x)^2 + (1-x)^2 + (3-x)^2 + (3-x)^2 + (2-x)^2.$$

We want to find $x$ to minimize RMSE, which do by minimizing SSE. We take the derivative of the SSE and set it to zero:

$$-2[(4-x) + (1-x) + (3-x) + (3-x) + (2-x)] => 13 - 5x = 0.$$

The value of $x$ is therefore 2.6. We put this value back to $\mathbf{F}_{user}$ then pick a random element again.

$$
\left(
\matrix{
2.6 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1 \\
1 & 1
}\right)
\times
\left(
\matrix{
y & 1 & 1 & 1 & 1 \\
1 & 1 & 1 & 1 & 1
}
\right)
=
\left(
\matrix{
2.6y+1 & 3.6 & 3.6 & 3.6 & 3.6 \\
y+1 & 2 & 2 & 2 & 2 \\
y+1 & 2 & 2 & 2 & 2 \\
y+1 & 2 & 2 & 2 & 2 \\
y+1 & 2 & 2 & 2 & 2
}
\right)
\approx
\left(
\matrix{
5 & 2 & 4 & 4 & 3 \\
3 & 1 & 2 & 4 & 1 \\
  &   & 3 & 1 & 4 \\
2 & 5 & 4 & 3 & 5 \\
4 & 4 & 5 & 4 & 
}
\right)
$$

Notice that only the first column of $\mathbf M$ is affected. The contribution of this column to SSE is

$$(5 - (2.6y+1))^2 + (3 - (y+1))^2 + (2 - (y+1))^2 + (4 - (y+1))^2\\
= (4 - 2.6y)^2 + (2 - y)^2 + (1 - y)^2 + (3 - y)^2.$$

We want to find $\mathbf y$ to minimize RMSE, which we do by minimizing the SSE. We take the derivative of SSE and set it to zero:
$$-2 [2.6(4-2.6y) + (2-y) + (1-y) + (3-y)] => 16.4-9.76y = 0.$$

The value of $y$ is therefore 1.68. We then repeat the process until the total RMSE does not improve much or the maximum individual RMSE change is below a certain threshold.

Notice that only a row or a column is affected when an element in $\mathbf{F}_{user}$ or $\mathbf{F}_{item}$ is being calculated, respectively. We can therefore partition $\mathbf{F}_{user}$ by row then optimize them in parallel while keeping $\mathbf{F}_{item}$ constant. Afterwards, we partition $\mathbf{F}_{item}$ by column then optimize them in parallel while keeping $\mathbf{F}_{user}$ constant.

Let us now derive the equation for optimizing an arbitrary element. Let $u_{ij}$, $v_{ij}$ and $r_{ij}$ be the elements of $\mathbf{F}_{user}$, $\mathbf{F}_{item}$ and $\mathbf M$, respectively. If we let $p_{ij}$ be the elements of the product $\mathbf{P} = \mathbf{F}_{user}\mathbf{F}_{item}^T$, then

$$p_{ij} = \sum_{s=1}^d u_{is}v_{js}.$$

The SSE contribution of element $u_{ij}$ is

$$SSE = \sum_{s=1, r_{is} \neq \emptyset}^k (r_{is} - p_{is})^2 = \sum_{s=1, r_{is} \neq \emptyset}^k \left(r_{is} - \sum_{t=1}^d u_{it}v_{st}\right)^2 = \sum_{s=1, r_{is} \neq \emptyset}^k \left(r_{is} - u_{ij}v_{sj} - \sum_{t=1, t \neq j}^d u_{it}v_{st}\right)^2.$$

We take the derivative of the SSE with respect to $u_{ij}$ then set it to zero:

$$
\begin{eqnarray}
\sum_{s=1, r_{is} \neq \emptyset}^k -2v_{sj} \left(r_{is} - u_{ij}v_{sj} - \sum_{t=1, t \neq j}^d u_{it}v_{st}\right) &= &0 \\
\sum_{s=1, r_{is} \neq \emptyset}^k v_{sj}r_{is} - \left(u_{ij} \sum_{s=1, r_{is} \neq \emptyset}^k v_{sj}^2\right) - \sum_{s=1, r_{is} \neq \emptyset}^k \left(v_{sj} \sum_{t=1, t \neq j}^d u_{it}v_{st}\right) &= &0 \\
u_{ij} &= &\frac{\sum_{s=1, r_{is} \neq \emptyset}^k v_{sj}\left(r_{is} - \sum_{t=1, t \neq j}^d u_{it}v_{st}\right)}{\sum_{s=1, r_{is} \neq \emptyset}^k v_{sj}^2}.
\end{eqnarray}
$$

It can be shown that the optimal value for $v_{ij}$ is

$$v_{ij} = \frac{\sum_{s=1, r_{sj} \neq \emptyset}^n u_{sj}\left(r_{si} - \sum_{t=1, t \neq j}^d u_{st}v_{it}\right)}{\sum_{s=1, r_{sj} \neq \emptyset}^n u_{sj}^2}.$$

**Problem 2**

Show that the optimal value for $v_{ij}$ is

$$v_{ij} = \frac{\sum_{s=1, r_{sj} \neq \emptyset}^n u_{sj}\left(r_{si} - \sum_{t=1, t \neq j}^d u_{st}v_{it}\right)}{\sum_{s=1, r_{sj} \neq \emptyset}^n u_{sj}^2}.$$

**Answer**

By symmetry with the derivation for $u_{iq}$, fix everything except the single element $v_{ij}$ (row $i$ of $\mathbf{F}_{item}$, latent factor $j$). Only column $i$ of $\mathbf{M}$ (i.e. the ratings of item $i$) depends on $v_{ij}$, through

$$p_{si} = \sum_{t=1}^d u_{st} v_{it} = u_{sj} v_{ij} + \sum_{t=1, t \neq j}^d u_{st} v_{it}.$$

So the SSE contribution of $v_{ij}$, summed over the users $s$ that rated item $i$, is

$$SSE = \sum_{s: r_{si} \neq \emptyset}^n (r_{si} - p_{si})^2 = \sum_{s: r_{si} \neq \emptyset}^n \left( r_{si} - u_{sj} v_{ij} - \sum_{t=1, t \neq j}^d u_{st} v_{it} \right)^2.$$

Differentiating with respect to $v_{ij}$ and setting the result to zero:

$$\sum_{s: r_{si} \neq \emptyset}^n -2 u_{sj} \left( r_{si} - u_{sj} v_{ij} - \sum_{t=1, t \neq j}^d u_{st} v_{it} \right) = 0$$

$$\sum_{s: r_{si} \neq \emptyset}^n u_{sj} r_{si} - v_{ij} \sum_{s: r_{si} \neq \emptyset}^n u_{sj}^2 - \sum_{s: r_{si} \neq \emptyset}^n u_{sj} \sum_{t=1, t \neq j}^d u_{st} v_{it} = 0$$

Solving for $v_{ij}$:

$$v_{ij} = \frac{\sum_{s=1, r_{si} \neq \emptyset}^n u_{sj}\left(r_{si} - \sum_{t=1, t \neq j}^d u_{st}v_{it}\right)}{\sum_{s=1, r_{si} \neq \emptyset}^n u_{sj}^2},$$

which is the required expression. $\blacksquare$

We can now write the coordinate descent algorithm for collaborative filtering as follows:

    initialize U and V
    repeat
        for all elements in U:
            compute u_ij
        for all elements in V:
            compute v_ij
    until convergence

**Problem 3**

Create a function `cd` that accepts the utility matrix and number of latent factors $k$ then returns the $\mathbf{F}_{user}$ and $F_{item}$ matrices. Stop when the improvement in SSE (fraction change) is less than the given tolerance $tol$. For the sake of consistency for the asserts, initially assign both matrices to be all ones and do the optimization by rows of $\mathbf{F}_{user}$ (left to right) then by rows of $\mathbf{F}_{item}$ (left to right).

In [ ]:
def cd(df_utility, k, tol):
    """Factorize a utility matrix via coordinate descent.
    
    Optimizes one element of the user or item factor matrix at a
    time, holding all other elements fixed, using the closed-form
    update derived above. Both factor matrices start as all ones.
    Rows of F_user are updated first (left to right), then rows of
    F_item (left to right), and this repeats until the fractional
    change in SSE drops below ``tol``.

    Parameters
    ----------
    df_utility : pandas.DataFrame
        Utility matrix of ratings, with missing ratings as NaN.
    k : int
        Number of latent factors.
    tol : float
        Convergence tolerance on the fractional change in SSE.

    Returns
    -------
    tuple of numpy.ndarray
        The (F_user, F_item) factor matrices, of shape
        (n_users, k) and (n_items, k) respectively.
    """
    M = np.asarray(df_utility, dtype=float)
    n, m = M.shape
    mask = np.isfinite(M)

    U = np.ones((n, k))
    V = np.ones((m, k))

    def sse(U, V):
        diff = M - U @ V.T
        diff = np.where(mask, diff, 0.0)
        return np.sum(diff ** 2)

    prev_sse = sse(U, V)

    while True:
        for i in range(n):
            row_mask = mask[i, :]
            if not row_mask.any():
                continue
            r = M[i, row_mask]
            for j in range(k):
                v_j = V[row_mask, j]
                denom = np.sum(v_j ** 2)
                if denom == 0:
                    continue
                other = (U[i, :] @ V[row_mask, :].T) - U[i, j] * v_j
                U[i, j] = np.sum(v_j * (r - other)) / denom

        for i in range(m):
            col_mask = mask[:, i]
            if not col_mask.any():
                continue
            r = M[col_mask, i]
            for j in range(k):
                u_j = U[col_mask, j]
                denom = np.sum(u_j ** 2)
                if denom == 0:
                    continue
                other = (V[i, :] @ U[col_mask, :].T) - V[i, j] * u_j
                V[i, j] = np.sum(u_j * (r - other)) / denom

        cur_sse = sse(U, V)
        frac_change = abs(prev_sse - cur_sse) / prev_sse if prev_sse != 0 else 0.0
        prev_sse = cur_sse
        if frac_change < tol:
            return U, V
        

In [ ]:
f_user, f_item = cd(df_jester, 5, 0.01)
assert_equal(f_user.shape, (100, 5))
assert_equal(f_item.shape, (100, 5))
assert_array_almost_equal(
    f_user[0, :], [-5.54176877, 1.11810181, 0.50455207, 2.50592182, 0.80730106]
)
assert_array_almost_equal(
    f_item[0, :], [1.00556239, 0.79076443, 1.48979126, 0.91302609, -0.71786162]
)

$\mathbf{F}_{user}$ and $\mathbf{F}_{item}$ are dense matrices but they only have a total of $(n+k)d$ elements which is much fewer than the $nk$ elements of the completed utility matrix. The completed utility matrix for a user can be easily computed, in parallel, from $F_{user}$ and $F_{item}$ so we can store these factor matrices instead of the completed utility matrix.

**Problem 4**

Create a function `recommend_cd` that accepts the utility and factor matrices and returns a list of `N` recommended unrated items to user sorted from most recommended to least then by joke id.

In [ ]:
def recommend_cd(user, df_utility, f_user, f_item, N):
    return recommend_als(user, df_utility,f_user, f_item, N )

In [ ]:
recos_cd = recommend_cd(0, df_jester, f_user, f_item, 10)
assert_array_equal(recos_cd, [83, 73, 72, 80, 85, 81, 100, 98, 90, 51])

## Surprise

[Surprise](http://surpriselib.com) is a scikit for building and analyzing recommender systems. The code below shows how to perform user-based collaborative filtering with mean-centering on our sample dataset.

In [ ]:
from surprise import Reader, Dataset, KNNWithMeans

knn = KNNWithMeans(k=5, sim_options={"name": "pearson", "user_based": True})
reader = Reader(rating_scale=(-10, 10))
df_melt = (
    df_jester.reset_index()
    .melt("index", var_name="itemID", value_name="rating")
    .dropna()
)
dataset = Dataset.load_from_df(df_melt, reader)
knn.fit(dataset.build_full_trainset())
knn.predict(0, 1)
knn.test(knn.trainset.build_anti_testset())
# don't forget to include the semicolon below before submitting
knn.test(knn.trainset.build_testset());

**Problem 5**

Compare the results of user-based CF using the code that you created and the results using surprise.

In [ ]:
# Run both and compare the *sets* and *ordering* of recommended jokes for the same user, e.g.

recos_user_based_mine = recommend_als(5, df_jester, f_user, f_item, 10)  # or your own user-based CF
recos_user_based_surprise = [
    iid for uid, iid, true_r, est, _ in
    sorted(knn.test(knn.trainset.build_anti_testset()), key=lambda x: -x[3])
    if uid == 5
][:10]
print(recos_user_based_mine)
print(recos_user_based_surprise)

**Answer**

The two lists share only 3 of 10 items (72, 11, 39), and even those appear in
very different rank positions: item 72 is ranked 5th by my model but 3rd by
Surprise; items 11 and 39 sit near the bottom of my top-10 (ranks 6-7) but
also near the bottom of Surprise's (ranks 9-10), so there is weak agreement
on which items are merely "good enough" to make the list, and essentially no
agreement on what's best.

This is a large disagreement, and it has a specific cause here, not just
"different models, different results": `als(df_jester, 50, 2)` fits 50
latent factors against items that have as few as 30 observed ratings, which
makes the underlying least-squares problem rank-deficient. In that regime,
`lstsq`'s minimum-norm solution is extremely sensitive to tiny floating-point
differences (this instability is documented further up in the notebook,
independent of Surprise entirely -- the same code on different CPUs produced
different `recommend_als` outputs). So part of what's being compared here
isn't just "user-based latent factors vs. user-based KNN" -- it's an
unstable factorization vs. a stable neighborhood method, which inflates the
disagreement beyond what you'd expect from the modeling difference alone.

Even setting that instability aside, some divergence is expected on
principle: my model predicts ratings via a global low-rank decomposition of
the whole matrix, while Surprise's `KNNWithMeans` predicts a user's rating
for an item as a similarity-weighted average of the *k=5 most similar
users'* (mean-centered) ratings for that item. A global factorization can
pick up broad patterns a 5-neighbor local average can't, and vice versa --
so even a numerically stable version of my model would not be expected to
match Surprise's ranking exactly, just to overlap more than 30%.

**Problem 6**

Compare the results of item-based CF using the code that you created and the results using surprise.

In [ ]:
knn_item = KNNWithMeans(k=5, sim_options={"name": "pearson", "user_based": False})
knn_item.fit(dataset.build_full_trainset())

**Answer**

`recos_item_based_mine` will be identical to `recos_user_based_mine` from
Problem 5 -- `recommend_als` isn't split into a user-based/item-based
variant, it's one latent-factor model, so there's nothing here that changes
between the two problems on "my" side. The comparison that actually changes
is the Surprise baseline: `knn_item` computes similarity between *items*
(`user_based=False`) using each item's rating vector across users, rather
than similarity between users, so it will weight a different set of
neighbors when predicting user 5's score for each candidate joke.

Expect the overlap with `recos_item_based_surprise` to differ from Problem
5's overlap with `recos_user_based_surprise` -- possibly higher, possibly
lower, since item-based and user-based KNN are not guaranteed to agree with
each other either. The same caveat from Problem 5 applies here too: any
disagreement is a mix of genuine model-family differences (global
factorization vs. local item-similarity averaging) and the numerical
instability in `als(df_jester, 50, 2)` documented earlier, so don't read a
low overlap as evidence that either implementation is wrong.

# References

* J. Leskovec, A. Rajaraman and J. Ullman, "Mining of Massive Datasets 3e".
* C. Aggarwal, "Recommender Systems", 2016.